In [1]:
# !python -m spacy download it_core_news_lg
# !pip install stanza

In [ ]:
import time
import numpy as np
import json
import stanza
from tqdm import tqdm
import uuid

import pandas as pd

stanza.download("it")  # once

In [3]:
class StanzaTokenizer:
    def __init__(self):
        self.nlp = stanza.Pipeline("it", processors="tokenize,pos,lemma,depparse")

    def tokenize(self, text):
        doc = self.nlp(text)
        tokens = []
        for sentence in doc.sentences:
            for token in sentence.tokens:
                word = token.words[0]
                tokens.append({
                    "text": token.text,
                    "lemma": word.lemma,
                    "pos": word.upos,
                    "xpos": word.xpos,
                    "head": word.head,
                    "deprel": word.deprel,
                    "vector": None,  # Stanza does not provide word vectors
                })
        return tokens

In [ ]:
tokenizer = StanzaTokenizer()

In [ ]:
path = '/Users/stevie/repos/lingo_kit_combined/lingo_kit_data/word_analysis/datasets/tatoeba/dataframe.tsv'
df = pd.read_csv(path, sep='\t')
len(df), df.columns

In [ ]:
df.head()

In [7]:
def get_token_hash(term, lemma, pos):
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{term}-{lemma}-{pos}"))

def get_group_hash(lemma, pos):
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{lemma}-{pos}"))

In [8]:
def normalize(text):
    text = text.lower()
    text = text.replace("’", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.strip()
    return text

In [ ]:
start_i = 40000
end_i = 60000
df = df.iloc[start_i:end_i]
print(f"Processing rows {start_i} to {end_i}")
print(f"Total rows: {len(df)}")

In [ ]:
print(df.head(n=5))
print(df[0:5])

In [ ]:
data = {}
for _, row in tqdm(df.iterrows(), total=len(df)):
    tokens = tokenizer.tokenize(row['text_it'])
    for token in tokens:
        word = normalize(token['text'])
        token_hash = get_token_hash(word, token['lemma'], token['pos'])
        group_hash = get_group_hash(token['lemma'], token['pos'])
        if token_hash not in data:
            data[token_hash] = {
                'word': word,
                'lemma': token['lemma'],
                'pos': token['pos'],
                'xpos': token['xpos'],
                'deprel': token['deprel'],
                'count': 0,
                'sentences': set(),
                'group_hash': group_hash,
            }
        data[token_hash]['count'] += 1
        data[token_hash]['sentences'].add(row['hash'])

In [12]:
token_df_data = {'token_hash': [], 'word': [], 'lemma': [], 'pos': [], 'xpos': [], 'deprel': [], 'count': [], 'sentences': [], 'group_hash': []}
for token_hash, info in data.items():
    if info['pos'] == 'PUNCT':
        continue
    token_df_data['token_hash'].append(token_hash)
    token_df_data['word'].append(info['word'])
    token_df_data['lemma'].append(info['lemma'])
    token_df_data['pos'].append(info['pos'])
    token_df_data['xpos'].append(info['xpos'])
    token_df_data['deprel'].append(info['deprel'])
    token_df_data['count'].append(info['count'])
    token_df_data['sentences'].append(list(info['sentences']))
    token_df_data['group_hash'].append(info['group_hash'])
token_df = pd.DataFrame(token_df_data)

In [13]:
token_df.sort_values(by='count', ascending=False, inplace=True)

In [14]:
save_path = f'/Users/stevie/repos/lingo_kit_combined/lingo_kit_data/word_analysis/datasets/tatoeba/token_data_{start_i}_{end_i}.tsv'
token_df.to_csv(save_path, sep='\t', index=False)

In [ ]:
token_df = pd.read_csv(save_path, sep='\t')
len(token_df), token_df.columns